In [4]:
import os

print('原始路径：' + os.getcwd())
os.chdir('/workspace/')
print('新路径：' + os.getcwd())


原始路径：/
新路径：/workspace


In [5]:
from unsloth import FastLanguageModel

max_seq_length = 1024
model_name = 'unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True
)



==((====))==  Unsloth 2025.8.5: Fast Qwen2 patching. Transformers: 4.55.4.
   \\   /|    Tesla V100-SXM2-32GB. Num GPUs = 1. Max memory: 31.739 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<string>:37: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.


model.safetensors:   0%|          | 0.00/1.81G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

In [7]:
inference_prompt = """以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
{}

### Response:
<think>{}
"""

FastLanguageModel.for_inference(model)

question = '男，28岁，程序员，最近一周每天工作到半夜，感觉头晕、脖子疼，有时候还恶心。'
formatedPrompt = inference_prompt.format(question, '')
print(formatedPrompt)

inputs = tokenizer([formatedPrompt], return_tensors='pt').to('cuda')
attention_mask = inputs.input_ids.ne(tokenizer.pad_token_id).to('cuda')

outputs = model.generate(
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1000,
    use_cache = True,
)


以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
男，28岁，程序员，最近一周每天工作到半夜，感觉头晕、脖子疼，有时候还恶心。

### Response:
<think>



In [15]:
print(inputs)
print(outputs)


response = tokenizer.batch_decode(outputs, skip_special_tokens=True)
#print(response)
splitedResponse = response[0].split('### Response:')
print(response[0].split('### Response:')[1])

{'input_ids': tensor([[151646,  87752,  99639,  38989,  53481,  88802,   9370, 109504,  90395,
          54387, 104133,  99553, 100642, 102285,  16744,   9370,  31196,   8997,
          14880, 110479, 104191, 112449,   9370, 104787,   3837,  23031,  60548,
          75882,  34859,   8997,  18493, 102104, 101056,  37945, 104857, 104107,
          75882,  86119,  90395, 104004,  46944,  17177,  64682,   9370, 104107,
         100178,   3837,  23031, 103944, 104493,   9370, 104913, 108487,  33108,
          43815, 102188,   1773,   1406,  14374,  29051,    510,  56568, 109182,
         104316, 101057,  96050, 104595, 113272,   5373, 105262,  47764,  33108,
         101899, 100367,  99522, 103926, 103524, 106289, 100032,   8997,  14880,
         102104,  87752, 104316,  86119,   3407,  14374,  15846,    510,  70108,
           3837,     17,     23,  92015,   3837, 118552,   3837, 104044, 105309,
         101922,  99257,  26939, 111001,   3837, 100681, 116280,   5373, 107966,
         10090

In [18]:
# 模型训练的 Prompt 模板
train_prompt = """以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
{}

### Response:
<think>
{}
</think>
{}
"""

EOS_TOKEN = tokenizer.eos_token # 添加 EOS Token

def formatting_prompts_func(examples):
    inputs = examples["Question"]
    cots = examples["Complex_CoT"]
    outputs = examples["Response"]
    texts = []
    for input, cot, output in zip(inputs, cots, outputs):
        # 将 EOS Token 添加到样本最后
        text = train_prompt.format(input, cot, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset

dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "zh", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)


In [20]:
from IPython.display import display, Markdown

display(Markdown(dataset[0]["text"])) 

以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
根据描述，一个1岁的孩子在夏季头皮出现多处小结节，长期不愈合，且现在疮大如梅，溃破流脓，口不收敛，头皮下有空洞，患处皮肤增厚。这种病症在中医中诊断为什么病？

### Response:
<think>
这个小孩子在夏天头皮上长了些小结节，一直都没好，后来变成了脓包，流了好多脓。想想夏天那么热，可能和湿热有关。才一岁的小孩，免疫力本来就不强，夏天的湿热没准就侵袭了身体。

用中医的角度来看，出现小结节、再加上长期不愈合，这些症状让我想到了头疮。小孩子最容易得这些皮肤病，主要因为湿热在体表郁结。

但再看看，头皮下还有空洞，这可能不止是简单的头疮。看起来病情挺严重的，也许是脓肿没治好。这样的情况中医中有时候叫做禿疮或者湿疮，也可能是另一种情况。

等一下，头皮上的空洞和皮肤增厚更像是疾病已经深入到头皮下，这是不是说明有可能是流注或瘰疬？这些名字常描述头部或颈部的严重感染，特别是有化脓不愈合，又形成通道或空洞的情况。

仔细想想，我怎么感觉这些症状更贴近瘰疬的表现？尤其考虑到孩子的年纪和夏天发生的季节性因素，湿热可能是主因，但可能也有火毒或者痰湿造成的滞留。

回到基本的症状描述上看，这种长期不愈合又复杂的状况，如果结合中医更偏重的病名，是不是有可能是涉及更深层次的感染？

再考虑一下，这应该不是单纯的瘰疬，得仔细分析头皮增厚并出现空洞这样的严重症状。中医里头，这样的表现可能更符合‘蚀疮’或‘头疽’。这些病名通常描述头部严重感染后的溃烂和组织坏死。

看看季节和孩子的体质，夏天又湿又热，外邪很容易侵入头部，对孩子这么弱的免疫系统简直就是挑战。头疽这个病名听起来真是切合，因为它描述的感染严重，溃烂到出现空洞。

不过，仔细琢磨后发现，还有个病名似乎更为合适，叫做‘蝼蛄疖’，这病在中医里专指像这种严重感染并伴有深部空洞的情况。它也涵盖了化脓和皮肤增厚这些症状。

哦，该不会是夏季湿热，导致湿毒入侵，孩子的体质不能御，其病情发展成这样的感染？综合分析后我觉得‘蝼蛄疖’这个病名真是相当符合。
</think>
从中医的角度来看，你所描述的症状符合“蝼蛄疖”的病症。这种病症通常发生在头皮，表现为多处结节，溃破流脓，形成空洞，患处皮肤增厚且长期不愈合。湿热较重的夏季更容易导致这种病症的发展，特别是在免疫力较弱的儿童身上。建议结合中医的清热解毒、祛湿消肿的治疗方法进行处理，并配合专业的医疗建议进行详细诊断和治疗。
<｜end▁of▁sentence｜>

In [22]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing="unsloth",
    random_state=1432,
    use_rslora=False,
    loftq_config=None,
)

print(model)

/root/miniforge3/lib/python3.11/site-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
Unsloth 2025.8.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
       

In [30]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = 'text',
    max_seq_length = 1024,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 6,
        gradient_accumulation_steps = 2,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 1432,
        output_dir = "outputs",
        report_to = "none",
    )
)


Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [31]:
import torch

trainer_stats = trainer.train()

print(trainer_stats)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20,171 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 6 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (6 x 2 x 1) = 12
 "-____-"     Trainable parameters = 18,464,768 of 1,795,552,768 (1.03% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,entropy
1,3.215500,0
2,3.258300,No Log
3,3.191200,No Log
4,3.197900,No Log
5,3.118300,No Log
6,3.116700,No Log
7,2.973300,No Log
8,2.773900,No Log
9,2.806900,No Log
10,2.805600,No Log


TrainOutput(global_step=60, training_loss=2.473056487242381, metrics={'train_runtime': 116.1443, 'train_samples_per_second': 6.199, 'train_steps_per_second': 0.517, 'total_flos': 5363235697692672.0, 'train_loss': 2.473056487242381, 'epoch': 0.03569303985722784})


In [32]:
model.save_pretrained("qwen-1.5b_lora_model")
tokenizer.save_pretrained("qwen-1.5b_lora_model")

('qwen-1.5b_lora_model/tokenizer_config.json',
 'qwen-1.5b_lora_model/special_tokens_map.json',
 'qwen-1.5b_lora_model/chat_template.jinja',
 'qwen-1.5b_lora_model/tokenizer.json')

In [33]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

question="一个患有急性阑尾炎的病人已经发病5天，腹痛稍有减轻但仍然发热，在体检时发现右下腹有压痛的包块，此时应如何处理？", # Question
inputs = tokenizer([inference_prompt.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1000,
    use_cache=True,
)

In [34]:
output = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print(output[0].split("### Response:")[1])


<think>
这位病人已经患病了，急性阑尾炎，已经5天了。腹痛好像只是轻微的减轻，但发热却明显。在检查的时候，右下腹突然出现了一个压痛的包块，这让我有点紧张。

首先，这个压痛包块是有可能导致感染的。如果感染严重，可能会有感染性休克。而如果是非感染性的疼痛，可能是因为炎症引起的，需要进一步确认。

考虑到急性阑尾炎的常见病史，这个压痛包块的可能性很高。尤其是当患者出现发热和腹痛时，这可能是感染性休克的表现。

但是，如果包块看起来不像是感染性的，而更像是炎症性疼痛，那可能需要更仔细地检查，比如使用非侵入性方法来确认。

再想想，急性阑尾炎患者通常会伴随腹痛和发热，这些症状可能暗示感染性休克。因此，这个压痛包块很可能与感染性休克有关。

不过，如果包块看起来不像是感染性的，可能需要考虑是否有其他因素，比如感染性疼痛或其他并发症。

总之，首先需要确认这个包块是否真的有感染性。如果没有感染，可能需要进一步的检查，以确保包块不是非感染性的。

所以，首先应该进行非侵入性的疼痛检查，比如使用X光或CT扫描，以确认包块的类型。

如果包块是感染性，那么可能需要立即进行感染性休克的处理，比如使用抗生素治疗和血液透析。
</think>
根据病人的描述，病人已经患有了急性阑尾炎，并且已经5天了，腹痛轻微减轻但仍有发热。在检查时，右下腹突然出现了一个压痛包块。这种症状很可能与感染性休克有关。

首先，需要确认这个包块是否真的是感染性。如果包块是感染性，应该立即进行感染性休克的处理，通常包括使用抗生素治疗和血液透析。如果没有感染性，可能需要考虑非感染性的疼痛。

因此，最合理的做法是进行非侵入性的疼痛检查，比如使用X光或CT扫描来确认包块的类型。如果包块是感染性的，立即进行感染性休克的处理。如果包块不是感染性的，可能需要进一步的检查或治疗。



In [36]:
def generate_response(question: str, model, tokenizer, inference_prompt: str, max_new_tokens: int = 1024) -> str:
    """
    使用指定的模型和分词器为给定的医学问题生成响应。

    Args:
        question (str): 需要模型回答的医学问题。
        model: 已加载的 Unsloth/Hugging Face 模型。
        tokenizer: 对应的分词器。
        inference_prompt (str): 用于格式化输入的 f-string 模板。
        max_new_tokens (int, optional): 生成响应的最大 token 数量。默认为 1024。

    Returns:
        str: 模型生成的响应文本，已去除 prompt 部分。
    """
    # 1. 使用模板格式化输入
    prompt = inference_prompt.format(
        question, # 填充问题
        "",       # 留空，让模型生成 CoT 和 Response
    )

    # 2. 将格式化后的 prompt 进行分词，并转移到 GPU
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    # 3. 使用模型生成输出
    # use_cache=True 用于加速解码过程
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens,
        use_cache=True,
    )
    
    # 4. 将生成的 token 解码为文本
    # skip_special_tokens=True 会移除像 EOS_TOKEN 这样的特殊标记
    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # 5. 切分字符串，只返回 "### Response:" 之后的部分
    # 使用 .split() 分割并获取响应内容，.strip() 用于去除可能存在的前后空白字符
    response_part = decoded_output.split("### Response:")
    if len(response_part) > 1:
        return response_part[1].strip()
    else:
        # 如果模型没有生成 "### Response:" 标记，则返回整个生成内容以供调试
        return decoded_output

In [37]:
my_question = "对于一名60岁男性患者，出现右侧胸疼并在X线检查中显示右侧肋膈角消失，诊断为肺结核伴右侧胸腔积液，请问哪一项实验室检查对了解胸水的性质更有帮助？"

response = generate_response(my_question, model, tokenizer, inference_prompt)
print("==================== 模型回答 ====================")
print(response)

==================== 模型回答 ====================
<think>
这位60岁男性患者出现了右侧胸疼，并且在X线检查中发现右侧肋膈角消失，这种现象提示他可能有肺结核。肺结核的常见症状就是胸痛，而肺结核患者通常也会出现胸腔积液。但是，积液的性质和程度需要通过一些实验室检查来确定。

根据我的医学知识，肺结核患者最常见的是肺结核，这种病通常会导致肺部肿大，尤其是胸腔积液。不过，对于胸腔积液的性质，我需要考虑一下。肺结核患者通常会有明显的胸腔积液，但这种积液的性质其实和肺结核患者本身有所不同。我记得在肺结核的患者中，胸腔积液通常会比较大，而且这些积液的大小和性质可能与肺结核患者本身有关。

所以，胸腔积液的性质在肺结核患者中是重要的，因为这些患者通常会有大的、比较明显的胸腔积液。这种积液的性质可能和肺结核患者的肺部结构有关，因此在诊断时，这些性质的检查会很有帮助。
</think>
在肺结核患者中，胸腔积液的性质是重要的诊断依据。肺结核患者通常会有较大的胸腔积液，这些积液的性质与肺结核患者本身有一定的关联。因此，胸腔积液的性质检查在诊断肺结核患者时非常重要，因为这些性质可以帮助医生更好地理解患者的胸腔情况，并制定相应的治疗方案。


In [39]:
my_question = "对于一名 28 岁的男性患者，工作是程序员，常年熬夜，最近突然感觉头晕目眩，甚至有点恶心。请问有可能是什么疾病？"

response = generate_response(my_question, model, tokenizer, inference_prompt, 512)
print("==================== 模型回答 ====================")
print(response)

==================== 模型回答 ====================
<think>
这位28岁的男性，工作是程序员，工作时间长期熬夜，这种持续的高强度工作可能对他的身体造成负担。这种持续的高强度工作常常会导致疲劳，可能对脑部的供血和神经功能造成影响。头晕和恶心这些症状可能与脑供血不足有关，这可能与缺血性贫血有关。

脑供血不足通常会导致一些常见症状，比如头晕、恶心、疲劳、视力模糊、疲倦、紧张等。这些症状在患者中很常见，所以他们怀疑可能是缺血性贫血。

缺血性贫血是因为血红蛋白含量低，导致红细胞减少，从而影响脑部供血。这种情况下，患者通常会有头晕、恶心、疲劳等症状。

此外，还有一种可能性是脑水肿。脑水肿是由于缺血性贫血导致的，它会压迫神经，增加脑水肿的风险。

脑水肿的危险性很高，尤其是当患者出现脑水肿和急性脑水肿时，严重并发症的风险会增加。因此，如果患者出现这些症状，他们应该立即咨询医生，并评估脑水肿的风险。

在进行进一步的检查和治疗之前，建议患者立即就医，进行脑水肿的评估和检查。这可能包括血液检查、血液透析、脑水肿影像学检查等。

总的来说，这种持续的高强度工作和出现头晕、恶心等症状，提示患者可能患有缺血性贫血或脑水肿。如果这些症状持续存在，建议立即就医，并进行相应的检查和治疗。
</think>
这位28岁的男性，工作是程序员，长期熬夜，这种高强度的工作可能对他的身体造成负担。这种持续的高强度工作常常会导致疲劳，可能对脑部的供血和神经功能造成影响。头晕、恶心这些症状可能与脑供血不足有关，这可能与缺血性贫血有关。

缺血性贫血通常会导致一些常见症状，比如头晕、恶心、疲劳、视力模糊、疲倦、紧张等。这些症状在患者中很常见，所以他们怀疑可能是缺血性贫血。

缺血性贫血的患者通常会有头晕、恶心、疲劳等症状。此外，还有一种可能性是脑水肿，脑水肿是由于缺血性贫血导致的，它会压迫神经，增加脑水肿的风险。

脑水肿的危险性很高，尤其是当患者出现脑水肿和急性脑水肿时，严重并发症的风险会增加。因此，如果患者出现这些症状，他们应该立即就医，并评估脑水肿的风险。

在进行进一步的检查和治疗


In [40]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla V100-SXM2-32GB. Max memory = 31.739 GB.
26.654 GB of memory reserved.


In [41]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

Peak reserved memory = 26.654 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 83.979 %.
Peak reserved memory for training % of max memory = 0.0 %.
